In [22]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Attention from scratch
Goal: understand self-attention by solving the "first digit" memory task
that RNN / LSTM / CNN struggled with.

In [23]:
def make_memory_data(n, length=30):
    """Random digits 0-9; label is 1 if the FIRST digit is > 4."""
    x = torch.randint(0, 10, (n, length))   # (n, length)
    y = (x[:, 0] > 4).long()                # (n,)
    return x, y

In [24]:
x, y = make_memory_data(5)
print(x.shape, y.shape)
print(x[:, 0])
print(y)

torch.Size([5, 30]) torch.Size([5])
tensor([2, 2, 6, 7, 9])
tensor([0, 0, 1, 1, 1])


## A2. Attention by hand
Each word compares itself with every word (dot product),
softmax turns scores into weights (sum = 1),
the new vector is a weighted average of all vectors.

In [25]:
def toy_attention(x, idx):
    """Attention for ONE word (row idx of x) over all words in x."""
    scores = x @ x[idx]                      # similarity with every word, shape (n_words,)
    weights = torch.softmax(scores, dim=0)   # weights sum to 1
    new_vec = weights @ x                    # weighted average of all vectors
    return scores, weights, new_vec

# toy vectors: [how "alive", how "place/object"]
words = ["kot", "ulice", "byl"]
x_toy = torch.tensor([[1.0, 0.1],    # kot
                      [0.1, 1.0],    # ulice
                      [0.9, 0.2]])   # byl: looks for something alive

scores, weights, new_vec = toy_attention(x_toy, idx=2)
print("scores :", scores)
print("weights:", weights, "sum =", weights.sum().item())
print("new byl:", new_vec)

scores : tensor([0.9200, 0.2900, 0.8500])
weights: tensor([0.4057, 0.2161, 0.3783]) sum = 1.0
new byl: tensor([0.7677, 0.3323])


## A3. Self-attention with learned Q, K, V
Each word is projected three times (Linear layers):
- Q (query): what am I looking for?
- K (key): what do I offer as a label?
- V (value): what do I pass on if someone attends to me?
scores = Q K^T / sqrt(d) -> softmax -> weighted average of V.

In [26]:
def self_attention(x, w_q, w_k, w_v):
    """Single-head self-attention. x has shape (n_words, d)."""
    q = w_q(x)                          # (n_words, d) what each word is looking for
    k = w_k(x)                          # (n_words, d) label each word offers
    v = w_v(x)                          # (n_words, d) content each word passes on
    d = q.shape[-1]
    scores = q @ k.T / math.sqrt(d)     # (n_words, n_words) everyone vs everyone
    weights = torch.softmax(scores, dim=-1)   # each ROW sums to 1
    return weights @ v, weights         # new vectors (n_words, d) and the weights


torch.manual_seed(0)
x = torch.randn(3, 4)                   # 3 words, 4 numbers each (random for now)
w_q, w_k, w_v = (nn.Linear(4, 4, bias=False) for _ in range(3))

out, weights = self_attention(x, w_q, w_k, w_v)
print("out shape    :", out.shape)
print("weights shape:", weights.shape)
print(weights)
print("row sums     :", weights.sum(dim=-1))

out shape    : torch.Size([3, 4])
weights shape: torch.Size([3, 3])
tensor([[0.3484, 0.3430, 0.3087],
        [0.1911, 0.4861, 0.3228],
        [0.2104, 0.4680, 0.3216]], grad_fn=<SoftmaxBackward0>)
row sums     : tensor([1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


## A4. Attention on the "first digit" task
Embedding + position -> self-attention -> read the LAST token -> Linear.
If it works, the last token must learn to look at position 0.

In [27]:
class AttentionNet(nn.Module):
    """Embedding (+ position) -> one self-attention layer -> classify from LAST token."""

    def __init__(self, vocab_size=10, d=32, length=10, use_pos=True):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d)
        self.pos = nn.Embedding(length, d) if use_pos else None   # which position is it?
        self.w_q = nn.Linear(d, d, bias=False)
        self.w_k = nn.Linear(d, d, bias=False)
        self.w_v = nn.Linear(d, d, bias=False)
        self.fc = nn.Linear(d, 2)
        self.last_weights = None                                   # saved for inspection

    def forward(self, x):                                          # x: (batch, length)
        h = self.emb(x)                                            # (batch, length, d)
        if self.pos is not None:
            h = h + self.pos(torch.arange(x.shape[1], device=x.device))
        q, k, v = self.w_q(h), self.w_k(h), self.w_v(h)            # (batch, length, d)
        scores = q @ k.transpose(1, 2) / math.sqrt(q.shape[-1])    # (batch, length, length)
        weights = torch.softmax(scores, dim=-1)
        self.last_weights = weights.detach()
        out = weights @ v                                          # (batch, length, d)
        return self.fc(out[:, -1])                                 # only the last token


def make_loaders(n_train=20000, n_val=2000, length=10, batch_size=64):
    x_tr, y_tr = make_memory_data(n_train, length)
    x_va, y_va = make_memory_data(n_val, length)
    train_loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_va, y_va), batch_size=batch_size)
    return train_loader, val_loader


def run_epoch(model, loader, optimizer=None):
    """One pass over the data; trains if an optimizer is given. Returns (loss, acc)."""
    training = optimizer is not None
    model.train(training)
    loss_fn = nn.CrossEntropyLoss()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


def train_attention(use_pos=True, length=10, epochs=10, lr=3e-3):
    torch.manual_seed(42)
    train_loader, val_loader = make_loaders(length=length)
    model = AttentionNet(length=length, use_pos=use_pos).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        print(f"Epoch {epoch}/{epochs} | train acc {tr_acc:.3f} | val loss {va_loss:.3f} acc {va_acc:.3f}")
    return model, val_loader


model, val_loader = train_attention(use_pos=True)

# Where does the LAST token look? (weights of the last row, first validation sample)
xb, yb = next(iter(val_loader))
model.eval()
model(xb.to(device))
torch.set_printoptions(precision=2, sci_mode=False)
print(xb[0])
print(model.last_weights[0, -1])

Epoch 1/10 | train acc 0.971 | val loss 0.000 acc 1.000
Epoch 2/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 3/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 4/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 5/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 6/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 7/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 8/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 9/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 10/10 | train acc 1.000 | val loss 0.000 acc 1.000
tensor([7, 1, 3, 0, 6, 2, 9, 1, 2, 1])
tensor([1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00])


## A5. Positional embedding: why attention needs it

Self-attention treats the input as a *set*: it has no idea which digit came first.
Fix: add a learned vector per position, `h = emb(digit) + pos(position)`.

**Experiment:** same model, same data (length 10), with and without `pos`.

| model | val acc |
|---|---|
| attention + pos | 1.000 |
| attention, no pos | ~0.62 |

Why ~0.62 and not 0.5? Without order the model still sees *how many* digits are > 4.
The first digit is one of them, so more big digits make "first digit is big" more likely.

In [28]:
model_no_pos, _ = train_attention(use_pos=False)

Epoch 1/10 | train acc 0.615 | val loss 0.650 acc 0.608
Epoch 2/10 | train acc 0.628 | val loss 0.640 acc 0.614
Epoch 3/10 | train acc 0.630 | val loss 0.638 acc 0.623
Epoch 4/10 | train acc 0.631 | val loss 0.638 acc 0.624
Epoch 5/10 | train acc 0.629 | val loss 0.639 acc 0.619
Epoch 6/10 | train acc 0.632 | val loss 0.637 acc 0.619
Epoch 7/10 | train acc 0.631 | val loss 0.638 acc 0.622
Epoch 8/10 | train acc 0.629 | val loss 0.639 acc 0.618
Epoch 9/10 | train acc 0.631 | val loss 0.634 acc 0.622
Epoch 10/10 | train acc 0.632 | val loss 0.638 acc 0.626


In [29]:
class BaselineNet(nn.Module):
    """Embedding -> RNN / LSTM / CNN -> Linear(2). Same task and data as AttentionNet."""

    def __init__(self, kind, vocab_size=10, d=32):
        super().__init__()
        self.kind = kind
        self.emb = nn.Embedding(vocab_size, d)
        if kind == "rnn":
            self.core = nn.RNN(d, d, batch_first=True)
        elif kind == "lstm":
            self.core = nn.LSTM(d, d, batch_first=True)
        elif kind == "cnn":
            self.core = nn.Conv1d(d, d, kernel_size=3, padding=1)
        else:
            raise ValueError(kind)
        self.fc = nn.Linear(d, 2)

    def forward(self, x):                                  # x: (batch, length)
        h = self.emb(x)                                    # (batch, length, d)
        if self.kind == "cnn":
            h = torch.relu(self.core(h.transpose(1, 2)))   # Conv1d wants (batch, d, length)
            return self.fc(h.max(dim=2).values)            # global max pooling over positions
        out, _ = self.core(h)                              # (batch, length, d)
        return self.fc(out[:, -1])                         # last time step only


def set_forget_bias(lstm, value=2.0):
    """Set the forget-gate bias. PyTorch bias layout is [input | forget | cell | output]."""
    d = lstm.hidden_size
    with torch.no_grad():
        lstm.bias_ih_l0[d:2 * d].fill_(value)


def train_model(model, length, epochs=10, lr=3e-3, n_train=50000):
    """Train any of our models and return the best validation accuracy."""
    train_loader, val_loader = make_loaders(n_train=n_train, length=length)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_acc = 0.0
    for _ in range(epochs):
        run_epoch(model, train_loader, optimizer)
        _, val_acc = run_epoch(model, val_loader)
        best_acc = max(best_acc, val_acc)
    return best_acc


def compare_models(length=30, epochs=10):
    """Train every model on the same task and print the best val accuracy of each."""
    results = {}
    for name in ("rnn", "lstm", "lstm_bias", "cnn", "attention"):
        torch.manual_seed(42)
        if name == "attention":
            model = AttentionNet(length=length)
        else:
            model = BaselineNet("lstm" if name == "lstm_bias" else name)
        if name == "lstm_bias":
            set_forget_bias(model.core, 2.0)
        results[name] = train_model(model, length, epochs)
        print(f"{name:10s} best val acc: {results[name]:.3f}")
    return results


results_30 = compare_models(length=30)

rnn        best val acc: 0.519
lstm       best val acc: 0.512
lstm_bias  best val acc: 1.000
cnn        best val acc: 1.000
attention  best val acc: 1.000


no padding

In [30]:
torch.manual_seed(42)
cnn_valid = BaselineNet("cnn")
cnn_valid.core = nn.Conv1d(32, 32, kernel_size=3, padding=0)   # no padding at the edges
print(train_model(cnn_valid, length=30))

0.621


## A6. Comparison: RNN vs LSTM vs CNN vs attention

**Task:** random digits 0-9, label = 1 if the FIRST digit is > 4.
The model must remember one early digit until the end of the sequence.

| model | length 10 (val acc) | length 30 (val acc) |
|---|---|---|
| RNN | 1.00 | 1.00 |
| LSTM | 1.00 | 0.50 (stuck) |
| CNN (Conv1D + global max pooling) | 0.80 | 0.63 |
| Attention + pos | 1.00 | 1.00 |

(Keras runs; the length-30 attention result is filled in after running A6.)

**Why each one behaves this way**
- **RNN:** with enough data it can carry the first digit through all steps,
  but the gradient shrinks at every step (vanishing gradient), so it is unstable on long sequences.
- **LSTM:** the forget gate starts around 0.5, so after 29 steps the signal is
  about 0.5**29 ≈ 2e-9. Initializing the forget-gate bias to a positive value
  (e.g. 2.0) fixed it (val acc 1.000).
- **CNN:** global max pooling keeps only the strongest value per filter and drops
  positions, so it cannot say "the first digit".
- **Attention:** the last token looks directly at position 0 (weight ≈ 1.00),
  so distance does not matter. It needs a positional embedding, otherwise
  val acc drops to ~0.62.

**Trade-off:** attention builds an n × n weight table, so memory and compute
grow quadratically with sequence length.

In [31]:
model30, _ = train_attention(use_pos=True, length=30)

Epoch 1/10 | train acc 0.988 | val loss 0.000 acc 1.000
Epoch 2/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 3/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 4/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 5/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 6/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 7/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 8/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 9/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 10/10 | train acc 1.000 | val loss 0.000 acc 1.000


In [32]:
def show_last_token_scores(model, x):
    """Print the last token's score against every position (before softmax)."""
    model.eval()
    with torch.no_grad():
        h = model.emb(x) + model.pos(torch.arange(x.shape[1], device=x.device))
        q, k = model.w_q(h), model.w_k(h)
        scores = q @ k.transpose(1, 2) / math.sqrt(q.shape[-1])
    return scores[0, -1]          # last token vs each of the 10 positions


xb, yb = next(iter(val_loader))
print(xb[0])
print(show_last_token_scores(model, xb.to(device)))

tensor([7, 1, 3, 0, 6, 2, 9, 1, 2, 1])
tensor([11.30, -5.16, -3.14, -1.38,  3.48, -5.30, -0.73, -5.48, -0.73, -1.45])


# Part B: a small transformer (character-level GPT) on Pan Tadeusz

Goal: train a small model on the text of "Pan Tadeusz" (Adam Mickiewicz, public domain)
that predicts the next character, then generate new Polish-sounding text from it.

Pipeline:
text -> tokenizer -> IDs -> batches -> embedding (+ position)
-> [ causal self-attention -> feed-forward ] x N -> Linear -> next-character probabilities

| step | what we add |
|---|---|
| B1 | character tokenizer + data |
| B2 | batches (input and shifted target) |
| B3 | baseline model (no attention) |
| B4 | causal mask |
| B5 | multi-head attention |
| B6 | transformer block (feed-forward, residual, layer norm) |
| B7 | stacked blocks, training, text generation |

Why a different text than the video we watch: same steps, new data, so we can
check that we understand the concept and not just the code.

## B1. Character tokenizer and data

Token = one character. Vocabulary = all distinct characters in the text
(here it also includes Polish letters like ą, ę, ł, ó, ś, ż).

- Clean the raw file: the Wolne Lektury file contains a title header and a license/footer
  that are not part of the poem, so we cut them off.
- `stoi`: character -> ID, `itos`: ID -> character.
- `encode(text)` -> list of IDs, `decode(ids)` -> text.
- The whole text becomes one long tensor of IDs.
- Split: first 90% train, last 10% validation (text the model never trained on).

Vocabulary size: 445321 (fill in after running).

Why characters and not BPE: the vocabulary is tiny and there is nothing to train,
so we can focus on the transformer. The price is longer sequences.

In [33]:
import os
import urllib.request

URL = "https://wolnelektury.pl/media/book/txt/pan-tadeusz.txt"
PATH = "pan_tadeusz.txt"


def load_text(url=URL, path=PATH):
    """Download the text once (if missing) and return it as one string."""
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


text = load_text()
print("characters:", len(text))


characters: 447334


Clean the text 

In [34]:
def clean_text(raw):
    """Keep only the poem: cut the header before 'Litwo!' and the footer after '-----'."""
    start = raw.find("Litwo! Ojczyzno moja!")
    end = raw.rfind("-----")            # last separator line = start of the footer
    return raw[start:end].strip()

text = clean_text(load_text())
print("characters:", len(text))
print(repr(text[:60]))
print(repr(text[-60:]))

characters: 445321
'Litwo! Ojczyzno moja! ty jesteś jak zdrowie;\nIle cię trzeba '
'kami wieśniaczki usnuty\nZ modrych bławatków i zielonej ruty…'


Build dictionary of words

In [54]:
def build_vocab(text):
    """Return the sorted list of distinct characters and the two lookup dicts."""
    chars = sorted(set(text))                      # distinct characters, fixed order
    stoi = {ch: i for i, ch in enumerate(chars)}   # character -> ID
    itos = {i: ch for i, ch in enumerate(chars)}   # ID -> character
    return chars, stoi, itos


def encode(s, stoi):
    """Text -> list of IDs."""
    return [stoi[ch] for ch in s]


def decode(ids, itos):
    """List of IDs -> text."""
    return "".join(itos[i] for i in ids)

chars, stoi, itos = build_vocab(text)
ids = encode("Adam", stoi)
print(chars)
print(ids)
print(decode(ids, itos))
print(stoi)

['\n', ' ', '!', '(', ')', '*', ',', '-', '.', '/', '1', '2', '8', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '«', '»', 'Ó', 'à', 'æ', 'é', 'ó', 'ą', 'Ć', 'ć', 'ę', 'Ł', 'ł', 'ń', 'Ś', 'ś', 'Ź', 'ź', 'Ż', 'ż', '—', '…']
[16, 42, 39, 51]
Adam
{'\n': 0, ' ': 1, '!': 2, '(': 3, ')': 4, '*': 5, ',': 6, '-': 7, '.': 8, '/': 9, '1': 10, '2': 11, '8': 12, ':': 13, ';': 14, '?': 15, 'A': 16, 'B': 17, 'C': 18, 'D': 19, 'E': 20, 'F': 21, 'G': 22, 'H': 23, 'I': 24, 'J': 25, 'K': 26, 'L': 27, 'M': 28, 'N': 29, 'O': 30, 'P': 31, 'R': 32, 'S': 33, 'T': 34, 'U': 35, 'V': 36, 'W': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 

In [36]:
import torch


def make_splits(text, stoi, train_frac=0.9):
    """Encode the whole text into one tensor and split it into train / val."""
    data = torch.tensor(encode(text, stoi), dtype=torch.long)  # shape: (n_chars,)
    n_train = int(train_frac * len(data))                       # first 90% of positions
    return data, data[:n_train], data[n_train:]


data, train_data, val_data = make_splits(text, stoi)
print(data.shape, train_data.shape, val_data.shape)
print(data[:15])
print(decode(data[:15].tolist(), itos))

torch.Size([445321]) torch.Size([400788]) torch.Size([44533])
tensor([27, 47, 58, 61, 53,  2,  1, 30, 48, 41, 64, 63, 64, 52, 53])
Litwo! Ojczyzno


## B2. Batches

The model sees a window of `block_size` characters (the context) and predicts the next one.

- `x` = a random chunk of length `block_size`, shape `(batch, block_size)`.
- `y` = the same chunk shifted by one character (the targets).
- One chunk gives `block_size` training examples at once: at position t the input is
  characters 0..t and the target is character t+1.
- `get_batch(split)` picks random start positions from train or validation data.

Shapes: `x` and `y` are both `(batch, block_size)`.

In [37]:
def get_batch(data, block_size, batch_size):
    """Return a random batch: x (chunks) and y (the same chunks shifted by one char)."""
    # random start positions; must leave room for block_size + 1 chars
    starts = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in starts])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in starts])
    return x, y

xb, yb = get_batch(train_data, block_size=8, batch_size=4)
print(xb.shape, yb.shape)
print(xb)
print(yb)

torch.Size([4, 8]) torch.Size([4, 8])
tensor([[51, 71, 45, 77,  1, 57, 47, 75],
        [41, 47, 43, 80, 52, 47, 53, 52],
        [47, 43, 52, 47, 77, 63,  1, 48],
        [ 1,  1, 25, 59, 84,  1, 51, 47]])
tensor([[71, 45, 77,  1, 57, 47, 75,  1],
        [47, 43, 80, 52, 47, 53, 52, 63],
        [43, 52, 47, 77, 63,  1, 48, 72],
        [ 1, 25, 59, 84,  1, 51, 47, 43]])


how it works?

In [38]:
print(repr(decode(xb[2].tolist(), itos)))
print(repr(decode(yb[2].tolist(), itos)))

'ieniły j'
'eniły ją'


## B3. Baseline: bigram model

No attention yet. Each character looks only at itself: an embedding table of shape
`(vocab_size, vocab_size)` gives the scores (logits) for the next character directly.

- Loss: cross-entropy between logits and `y`.
- Sanity check: before training the loss should be about ln(vocab_size) (pure guessing).
- `generate(...)`: take the logits of the last position, softmax, sample one character,
  append it, repeat.

Baseline validation loss after training: 4.87.
Every later step should lower it, and the generated text should look more like Polish.

one initialization

In [39]:
import torch.nn as nn
import torch.nn.functional as F


class BigramModel(nn.Module):
    """Predict the next char from the current char only (a lookup table)."""

    def __init__(self, vocab_size):
        super().__init__()
        # row = current char, columns = scores for every possible next char
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x, y=None):
        logits = self.table(x)                       # (batch, block, vocab)
        if y is None:
            return logits, None
        b, t, v = logits.shape
        loss = F.cross_entropy(logits.view(b * t, v), y.view(b * t))
        return logits, loss


vocab_size = len(chars)
model = BigramModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss.item())

torch.Size([4, 8, 87])
4.872867107391357


### Bigram model: what it does

**Idea:** the model sees one letter and predicts the next one using only that letter. It is a lookup table.

- `nn.Embedding(vocab_size, vocab_size)` creates a table of 87 x 87 numbers.
  - Row = the letter the model **sees** (its number is the row index).
  - Column = a possible **next** letter.
- At the start the numbers are random, so the model guesses blindly (loss ≈ ln(87) ≈ 4.47).
- Training slowly changes the numbers: if "o" often follows "k" in the text, the cell (k, o) grows.

**Data flow**

| Step | Name | Shape | Meaning |
|---|---|---|---|
| 1 | `xb` | (batch, block) | letter numbers the model sees |
| 2 | `yb` | (batch, block) | correct next letters (`xb` shifted by 1) |
| 3 | `logits` | (batch, block, 87) | raw scores for every possible next letter |
| 4 | `loss` | single number | average error over all batch * block guesses |

**Why we need a table if `yb` has the answers:** `yb` only gives the answer for one place in the text. The same letter is followed by different letters in different places, so the table stores the overall tendency. At generation time there is no `yb`, so the model relies on the table.

**Logits** are raw scores (not probabilities). Softmax turns them into probabilities that sum to 1.

**Result:** val loss ≈ 2.59 (plateau around 2.5), because one letter of context is not enough.

more initializations

In [40]:
def train_bigram(model, train_data, steps, block_size, batch_size, lr=1e-2):
    """Train the model for a number of steps; print the loss every 500 steps."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        xb, yb = get_batch(train_data, block_size, batch_size)  # new random batch
        _, loss = model(xb, yb)                                 # guess + loss
        optimizer.zero_grad()                                   # clear old corrections
        loss.backward()                                         # which cells go up / down
        optimizer.step()                                        # apply small corrections
        if step % 500 == 0:
            print(step, loss.item())
    return model


model = BigramModel(vocab_size)
train_bigram(model, train_data, steps=3000, block_size=8, batch_size=32)

0 5.0107598304748535
500 2.7277748584747314
1000 2.5890049934387207
1500 2.604912519454956
2000 2.594587564468384
2500 2.591883659362793


BigramModel(
  (table): Embedding(87, 87)
)

In [41]:
@torch.no_grad()   # no learning here, just writing
def generate(model, max_new_tokens, start_id=0):
    """Generate text one char at a time by sampling from the model's probabilities."""
    idx = torch.tensor([[start_id]], dtype=torch.long)       # shape (1, 1)
    for _ in range(max_new_tokens):
        logits, _ = model(idx)
        probs = F.softmax(logits[:, -1, :], dim=-1)           # scores -> percentages
        next_id = torch.multinomial(probs, num_samples=1)     # draw one char by chance
        idx = torch.cat([idx, next_id], dim=1)                # append and repeat
    return idx[0].tolist()


print(decode(generate(model, 300), itos))


 rakro orwile  krd, poństrażo d wa kieczyszani, by zy nalspszasiczą pstustra, si sta ż w: wakwi i
Pci Hrzecnałużew wi!» wych grólele gwśle, drcznied ito tóranaromu).
I te jały prsem pubiówczę, ie. tka Wiej dłą; ws n j tych;
Lew nesiąw, nie ki
 nienagł
Dy dy.
Najsachalkru,
Ra pa mosie zyneje wy!
Ch n


before and after training

In [42]:
untrained = BigramModel(vocab_size)
print(decode(generate(untrained, 200), itos))   # before training
print("-----")
print(decode(generate(model, 200), itos))       # after training


śV;a
8c«NAPe»ààvOTxĆdVu:-ńruŹæuV1ŚTæSęoASOO,—hŻgGFlŁŁwOfjz«EkévK,:Uą/—*ÓJ,ŚDęŻ:WKeq.wIćÓi/-E»àĆxtRc.VŻfO2ćDVłź1!:J,łT. (ĆhpŹ2ŻJ źræÓcfC18-ńjÓ;w»DjŹ!«F)x2àŹp«-ŻNT?cŻ*
źnIęZ!*LszćapllłæńK…v
æuà
…J,)AŹżI
-----

Mo iazyj pieli Ho wę i otem  cici wichonie otej bu: zę;  dzu;
 wsach, kłysiczbiewi zegdz, d zlakłuch Takodzcabę dz kozę,
I nasznogów w owięszymch j pijerysnie pli nimieschuck bry ! wspy n siadzgla rad


## B4. Causal mask

Self-attention lets every token look at every other token. For next-character
prediction that is cheating: token t would see the character it has to predict.

Fix: token t may attend only to positions <= t.

- Build a lower-triangular matrix of ones, shape `(T, T)`.
- Before softmax, set the scores of the future positions to `-inf`.
- After softmax those positions get weight 0, and every row still sums to 1.

Check: print the weights of one sequence, the upper triangle must be all zeros.

In [43]:
def average_previous(x):
    """For each position, average the vectors of all positions up to and including it."""
    t = x.shape[-2]                                   # number of positions
    mask = torch.tril(torch.ones(t, t))               # lower triangle of ones
    weights = mask / mask.sum(dim=1, keepdim=True)    # every row sums to 1
    return weights @ x                                # works for (t, c) and (b, t, c)


def causal_weights(t):
    """Attention weights with equal scores, but future positions masked out."""
    scores = torch.zeros(t, t)                         # same score for every pair
    mask = torch.tril(torch.ones(t, t))                # 1 = allowed, 0 = future
    scores = scores.masked_fill(mask == 0, float("-inf"))
    return F.softmax(scores, dim=-1)                   # each row -> percentages


print(causal_weights(3))

tensor([[1.00, 0.00, 0.00],
        [0.50, 0.50, 0.00],
        [0.33, 0.33, 0.33]])


example

In [59]:
x = torch.tensor([[1.0, 3.0],
                  [3.0, 1.0],
                  [5.0, 5.0]])

print(x.shape)  
print(causal_weights(3))
print(average_previous(x))

torch.Size([3, 2])
tensor([[1.00, 0.00, 0.00],
        [0.50, 0.50, 0.00],
        [0.33, 0.33, 0.33]])
tensor([[1., 3.],
        [2., 2.],
        [3., 3.]])


## B5. Multi-head attention

One attention head can focus on one kind of pattern. Several heads in parallel can
look for different things at the same time.

- Split the vector size `d` into `n_heads` parts of size `d / n_heads`.
- Each head has its own Q, K, V and its own attention weights.
- Concatenate the head outputs back to size `d` and pass them through one Linear layer.

Shapes: input `(batch, T, d)` -> each head `(batch, T, d / n_heads)` -> concat `(batch, T, d)`.

In [44]:
class Head(nn.Module):
    """One head of causal self-attention."""

    def __init__(self, d, head_size, block_size):
        super().__init__()
        self.q = nn.Linear(d, head_size, bias=False)
        self.k = nn.Linear(d, head_size, bias=False)
        self.v = nn.Linear(d, head_size, bias=False)
        # stored with the model, but not learned
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        t = x.shape[1]                                          # number of positions
        q, k, v = self.q(x), self.k(x), self.v(x)               # steps 3: Q, K, V
        scores = q @ k.transpose(-2, -1) / k.shape[-1] ** 0.5   # steps 4-5: Q·K, divide by sqrt
        scores = scores.masked_fill(self.mask[:t, :t] == 0, float("-inf"))   # step 6: mask
        weights = F.softmax(scores, dim=-1)                     # step 7: percentages
        return weights @ v                                      # step 8: mix the V vectors

head = Head(d=4, head_size=2, block_size=8)
x = torch.randn(1, 3, 4)          # 1 text, 3 positions, 4 numbers per position
print(head(x).shape)

torch.Size([1, 3, 2])


In [45]:
class MultiHead(nn.Module):
    """Several attention heads in parallel, concatenated and mixed by one Linear."""

    def __init__(self, d, n_heads, block_size):
        super().__init__()
        head_size = d // n_heads                       # d must divide evenly
        self.heads = nn.ModuleList(
            [Head(d, head_size, block_size) for _ in range(n_heads)]
        )
        self.proj = nn.Linear(d, d)                    # mixes the heads' results

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (batch, T, d)
        return self.proj(out)                                  # (batch, T, d)


mh = MultiHead(d=8, n_heads=4, block_size=8)
x = torch.randn(1, 3, 8)
print(mh(x).shape)

torch.Size([1, 3, 8])


## B6. Transformer block

One block = attention + feed-forward, wrapped with two stabilizers.

- Feed-forward: Linear(d -> 4d), non-linearity, Linear(4d -> d), applied to each token separately.
  Attention mixes information between tokens, feed-forward processes each token on its own.
- Residual connection: `x = x + sublayer(x)`. The input is added back, so gradients
  and the original signal have a shortcut.
- Layer norm: normalizes each token vector, applied before attention and before feed-forward.
- Dropout: randomly zeroes some values during training to reduce overfitting.

Shape does not change: `(batch, T, d)` in, `(batch, T, d)` out.

In [46]:
class FeedForward(nn.Module):
    """Per-token MLP: expand d -> 4d, ReLU, project back 4d -> d."""

    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 4 * d),   # step 1: expand
            nn.ReLU(),             # step 2: zero out negatives
            nn.Linear(4 * d, d),   # step 3: shrink back
        )

    def forward(self, x):
        # x: (batch, T, d) -> same shape out
        return self.net(x)

ff = FeedForward(d=4)
x = torch.randn(1, 3, 4)
print(ff(x).shape)

torch.Size([1, 3, 4])


In [47]:
class Block(nn.Module):
    """Transformer block: pre-norm attention and feed-forward, with dropout and residuals."""

    def __init__(self, d, n_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)                   # before attention
        self.sa = MultiHead(d, n_heads, block_size)  # communication
        self.ln2 = nn.LayerNorm(d)                   # before feed-forward
        self.ff = FeedForward(d)                     # per-token thinking
        self.drop = nn.Dropout(dropout)              # active only in training

    def forward(self, x):
        x = x + self.drop(self.sa(self.ln1(x)))   # normalize, attend, drop, add back
        x = x + self.drop(self.ff(self.ln2(x)))   # normalize, think, drop, add back
        return x

In [48]:
block = Block(d=8, n_heads=4, block_size=8)
x = torch.randn(1, 3, 8)
print(block(x).shape)

torch.Size([1, 3, 8])


In [49]:
ln = nn.LayerNorm(8)
x = torch.randn(1, 3, 8) * 100   # numbers with a large scale

print("before: std =", x.std().item())
print("after:  std =", ln(x).std().item())
print("after:  mean per letter =", ln(x).mean(dim=-1))

before: std = 83.78331756591797
after:  std = 1.0215078592300415
after:  mean per letter = tensor([[-0.00,  0.00, -0.00]], grad_fn=<MeanBackward1>)


## B7. Full model, training, generation

Model:
token embedding + position embedding -> N blocks -> final layer norm -> Linear(d -> vocab_size)

Training: AdamW, cross-entropy on `(logits, y)`, check train and validation loss every N steps.

| model | validation loss |
|---|---|
| bigram baseline (B3) | 2.59 |
| + attention (B4) | 3.0 |
| + multi-head, block, N blocks (B7) | 1.92 |

Generation: start from one character (or a prompt), repeat:
1. crop the context to the last `block_size` characters,
2. run the model, take the logits of the last position,
3. softmax, sample a character, append it.

Note: the model sees at most `block_size` characters back, because the position
embedding only has that many rows.

In [50]:
class MiniGPT(nn.Module):
    """Character-level GPT: embeddings + positions -> blocks -> logits."""

    def __init__(self, vocab_size, d, n_heads, n_blocks, block_size, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d)   # letter -> vector
        self.pos_emb = nn.Embedding(block_size, d)   # position -> vector
        self.blocks = nn.Sequential(
            *[Block(d, n_heads, block_size, dropout) for _ in range(n_blocks)]
        )                                            # n_blocks blocks in a row
        self.ln_f = nn.LayerNorm(d)                  # final layer norm
        self.head = nn.Linear(d, vocab_size)         # vector -> scores for each letter

    def forward(self, x, y=None):
        b, t = x.shape
        tok = self.tok_emb(x)                        # (b, t, d)
        pos = self.pos_emb(torch.arange(t, device=x.device))  # (t, d)
        h = tok + pos                                # letter + position
        h = self.blocks(h)                           # (b, t, d)
        h = self.ln_f(h)                             # (b, t, d)
        logits = self.head(h)                        # (b, t, vocab_size)

        if y is None:
            return logits, None
        loss = F.cross_entropy(logits.view(b * t, -1), y.view(b * t))
        return logits, loss

In [51]:
model = MiniGPT(vocab_size=87, d=64, n_heads=4, n_blocks=3, block_size=64)
x = torch.randint(0, 87, (2, 10))   # 2 windows, 10 random letters each
logits, loss = model(x, x)
print(logits.shape, loss.item())

torch.Size([2, 10, 87]) 4.56226921081543


In [52]:
import time


@torch.no_grad()
def estimate_loss(model, data, block_size, batch_size, iters=50):
    """Average loss over several random batches, with dropout off."""
    model.eval()
    losses = []
    for _ in range(iters):
        xb, yb = get_batch(data, block_size, batch_size)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


def train_model(model, train_data, val_data, steps, block_size, batch_size,
                lr=1e-3, eval_every=200):
    """Train the model; print train/val loss every eval_every steps."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    start = time.time()
    for step in range(steps + 1):
        if step % eval_every == 0:
            tr = estimate_loss(model, train_data, block_size, batch_size)
            va = estimate_loss(model, val_data, block_size, batch_size)
            print(f"step {step}: train {tr:.3f}, val {va:.3f}, {time.time() - start:.0f}s")
        if step == steps:
            break
        xb, yb = get_batch(train_data, block_size, batch_size)
        _, loss = model(xb, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model

In [61]:
model = MiniGPT(vocab_size=87, d=64, n_heads=4, n_blocks=3, block_size=64)
train_model(model, train_data, val_data, steps=5000,
            block_size=64, batch_size=32, eval_every=500)

step 0: train 4.662, val 4.660, 1s
step 500: train 2.482, val 2.487, 16s
step 1000: train 2.314, val 2.323, 32s
step 1500: train 2.186, val 2.213, 47s
step 2000: train 2.099, val 2.135, 62s
step 2500: train 2.031, val 2.070, 78s
step 3000: train 1.992, val 2.041, 93s
step 3500: train 1.947, val 1.993, 108s
step 4000: train 1.917, val 1.962, 124s
step 4500: train 1.885, val 1.950, 139s
step 5000: train 1.866, val 1.925, 154s


MiniGPT(
  (tok_emb): Embedding(87, 64)
  (pos_emb): Embedding(64, 64)
  (blocks): Sequential(
    (0): Block(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (sa): MultiHead(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (q): Linear(in_features=64, out_features=16, bias=False)
            (k): Linear(in_features=64, out_features=16, bias=False)
            (v): Linear(in_features=64, out_features=16, bias=False)
          )
        )
        (proj): Linear(in_features=64, out_features=64, bias=True)
      )
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (ff): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): ReLU()
          (2): Linear(in_features=256, out_features=64, bias=True)
        )
      )
      (drop): Dropout(p=0.1, inplace=False)
    )
    (1): Block(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=

In [62]:
@torch.no_grad()
def generate_gpt(model, prompt, stoi, max_new_tokens, block_size,
                 temperature=1.0, top_k=None):
    """Generate text; temperature scales randomness, top_k keeps the k best letters."""
    model.eval()
    idx = torch.tensor([encode(prompt, stoi)], dtype=torch.long)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]                 # crop context
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature         # scale scores
        if top_k is not None:
            kth = torch.topk(logits, top_k).values[:, -1:]          # k-th best score
            logits = logits.masked_fill(logits < kth, float("-inf"))  # drop the rest
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx[0].tolist()

In [63]:
out = generate_gpt(model, "Litwo! Ojczyzno", stoi, 300, BLOCK_SIZE,
                   temperature=0.8, top_k=10)
print(decode(out, itos))

NameError: name 'BLOCK_SIZE' is not defined

In [ ]:
torch.save(model.state_dict(), "minigpt_pan_tadeusz.pt")   # save all learned numbers

## Summary

- Built a character-level GPT on Pan Tadeusz step by step: tokenizer, batches,
  bigram baseline, causal mask, self-attention, multi-head attention,
  transformer block, full MiniGPT.
- Validation loss: bigram 2.59, mean-of-previous-letters ~3.0,
  MiniGPT 1.92 (~165k parameters, d=64, 3 blocks, block_size=64).
- Samples: the bigram writes random letter pairs; MiniGPT writes verse-like
  lines with real words, punctuation and quotes.
- Sampling: temperature ~0.8 with top-k 10 gives the best-looking text;
  low temperature without top-k can get stuck in whitespace loops.
- Limits: tiny model and dataset, no real understanding of meaning.
  Further gains need more data and parameters (watch the train/val gap).